In [1]:
import axelrod as axl
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt


In [2]:
R = 3
S = 0
P = 1

T_values = [3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]

games = [axl.Game(r=R, s=S, t=T, p=P) for T in T_values]

p = [axl.Fuzzy()] + [player() for player in axl.stewart_plotkin_strategies]
dffs = []

for g in games:
    dfs = []
    for i in range(0,5):
        t = axl.Tournament(p, game=g, turns=200, repetitions=1)
        results = t.play(build_results=True)
        
        df = pd.DataFrame(
                {
                    "Name": [x.name for x in p],
                    "#Wins": list(map(lambda x: sum(x), results.wins)),
                    "TotalScore": list(map(lambda x: sum(x), results.scores)),
                    "CooperationRating" : results.cooperating_rating,
                    "GoodPartnerRating" : results.good_partner_rating,
                    "EigenJesusRating" : results.eigenjesus_rating,
                    "EigenMosesRating" : results.eigenmoses_rating,
                })

        df = df.sort_values("TotalScore", ascending=False)
        df = df.reset_index(drop=True)

        dfs.append(df)
    dffs.append(dfs)

Playing matches:  11%|█         | 13/120 [00:13<00:57,  1.86it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:  12%|█▏        | 14/120 [00:15<01:22,  1.28it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:  12%|█▏        | 14/120 [00:14<01:18,  1.35it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:  11%|█         | 13/120 [00:12<00:55,  1.91it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S)

In [3]:
writer = pd.ExcelWriter('axelord_t_values_sp.xlsx', engine='xlsxwriter')

j = 0
for dfs in dffs:
    i = 0
    for df in dfs:
        df.to_excel(writer, sheet_name=(str(T_values[j]) + " (" + str(i) + ")"), index=False)
        i += 1
    j += 1

writer.close()

In [4]:
finalDFS = []
t = 3
for dfs in dffs:
    combined = pd.concat(dfs)
        
    agg = combined.groupby('Name').agg({
            '#Wins': 'sum',
            'TotalScore': 'sum',
            'CooperationRating': 'mean',
            'GoodPartnerRating': 'mean',
            'EigenJesusRating': 'mean',
            'EigenMosesRating': 'mean'
    }).reset_index()
        
        # Compute %Wins and AvgScore
    num_dfs = len(dfs)
    total_matches = (len(p)-1) * num_dfs
    agg['%Wins'] = agg['#Wins'] / total_matches
    agg['AvgScore'] = agg['TotalScore'] / total_matches
    agg['T'] = t
        
        # Rename columns
    agg = agg.rename(columns={
        'CooperationRating': 'AvgCoopRating',
        'GoodPartnerRating': 'AvgGP',
        'EigenJesusRating': 'AvgEJ',
        'EigenMosesRating': 'AvgEM'
    })
        
        # Keep only desired columns
    result = agg[['Name', '%Wins', 'AvgScore', 'AvgCoopRating', 'AvgGP', 'AvgEJ', 'AvgEM', 'T']]
    result = result.sort_values(by="AvgScore", ascending=False)

    finalDFS.append(result)
    t+=0.5

writer = pd.ExcelWriter('axelord_t_values_sp_FinalDFS.xlsx', engine='xlsxwriter')


j = 0
for df in finalDFS:
    df.to_excel(writer, sheet_name=("T = " + str(T_values[j])), index=False)
    j += 1

justAvg = pd.concat(finalDFS)

justAvg = justAvg[['Name', 'AvgScore', 'T']]

justAvg.to_excel(writer, sheet_name="FINAL", index=False)



writer.close()